# NeuroGolf ONNX validation

`outputs/taskXXX.onnx` を読み込み、各 task の pass/fail と推定スコアを確認する notebook。

In [1]:
from pathlib import Path
import copy
import importlib.util
import json
import math
import os
import re
import time
import traceback

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd

def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "neurogolf-2026").is_dir() and (path / "outputs").is_dir():
            return path
    raise FileNotFoundError("project root not found")


ROOT = find_project_root()
DATA_DIR = ROOT / "neurogolf-2026"
OUTPUT_DIR = ROOT / "outputs"
PROFILE_DIR = ROOT / "profiles" / "notebook"
MPL_CACHE_DIR = ROOT / ".matplotlib-cache"
PROFILE_DIR.mkdir(exist_ok=True)
MPL_CACHE_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

spec = importlib.util.spec_from_file_location(
    "neurogolf_utils",
    DATA_DIR / "neurogolf_utils" / "neurogolf_utils.py",
)
ng = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ng)
ng._NEUROGOLF_DIR = str(DATA_DIR) + "/"

FILESIZE_LIMIT = int(1.44 * 1024 * 1024)
onnx_files = sorted(OUTPUT_DIR.glob("task*.onnx"))
print(f"found {len(onnx_files)} ONNX files")
onnx_files

found 10 ONNX files


[PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task001.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task002.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task003.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task004.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task005.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task006.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task007.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task008.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task009.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/task010.onnx')]

In [ ]:
def task_num_from_path(path: Path) -> int:
    m = re.search(r"task(\d{3})\.onnx$", path.name)
    if not m:
        raise ValueError(f"unexpected ONNX filename: {path}")
    return int(m.group(1))


def load_examples(task_num: int):
    with open(DATA_DIR / f"task{task_num:03d}.json") as f:
        return json.load(f)


def verify_subset(session, examples):
    passed = 0
    failed = 0
    first_failure = None
    skipped = 0
    for idx, example in enumerate(examples):
        benchmark = ng.convert_to_numpy(example)
        if benchmark is None:
            skipped += 1
            continue
        try:
            actual = ng.run_network(session, benchmark["input"])
            if np.array_equal(actual, benchmark["output"]):
                passed += 1
            else:
                failed += 1
                if first_failure is None:
                    first_failure = {"idx": idx, "example": example, "actual": ng.convert_from_numpy(actual)}
        except Exception as exc:
            failed += 1
            if first_failure is None:
                first_failure = {"idx": idx, "error": repr(exc), "traceback": traceback.format_exc()}
    return passed, failed, skipped, first_failure


def make_session_and_score(model, task_num):
    sanitized = ng.sanitize_model(copy.deepcopy(model))
    if sanitized is None:
        raise ValueError("sanitize_model failed")
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    options.profile_file_prefix = str(PROFILE_DIR / f"task{task_num:03d}")
    session = ort.InferenceSession(sanitized.SerializeToString(), options)
    return sanitized, session


def evaluate_onnx(path: Path):
    started = time.time()
    task_num = task_num_from_path(path)
    filesize = path.stat().st_size
    examples = load_examples(task_num)
    model = onnx.load(path)
    onnx.checker.check_model(model, full_check=True)
    sanitized, session = make_session_and_score(model, task_num)

    agi_pass, agi_fail, agi_skip, agi_failure = verify_subset(session, examples["train"] + examples["test"])
    gen_pass, gen_fail, gen_skip, gen_failure = verify_subset(session, examples["arc-gen"])
    profile_path = session.end_profiling()
    memory, params = ng.score_network(sanitized, profile_path)
    score = None
    if memory is not None and params is not None and memory >= 0 and params >= 0:
        score = max(1.0, 25.0 - math.log(max(1.0, memory + params)))

    return {
        "task": task_num,
        "onnx": str(path.relative_to(ROOT)),
        "filesize": filesize,
        "filesize_mb": filesize / 1024 / 1024,
        "size_ok": filesize <= FILESIZE_LIMIT,
        "arc_agi_pass": agi_pass,
        "arc_agi_fail": agi_fail,
        "arc_agi_skip": agi_skip,
        "arc_gen_pass": gen_pass,
        "arc_gen_fail": gen_fail,
        "arc_gen_skip": gen_skip,
        "ready": (filesize <= FILESIZE_LIMIT and agi_fail == 0 and gen_fail == 0),
        "memory_bytes": memory,
        "params": params,
        "estimated_score": score,
        "profile_path": profile_path,
        "seconds": time.time() - started,
        "first_failure": agi_failure or gen_failure,
    }


def evaluate_all(paths=onnx_files):
    rows = []
    failures = {}
    for path in paths:
        task_num = task_num_from_path(path)
        print(f"task {task_num:03d}: validating {path.name} ...", flush=True)
        try:
            row = evaluate_onnx(path)
            rows.append({k: v for k, v in row.items() if k != "first_failure"})
            if row["first_failure"] is not None:
                failures[task_num] = row["first_failure"]
            print(
                f"  ready={row['ready']} agi={row['arc_agi_pass']}/{row['arc_agi_pass'] + row['arc_agi_fail']} "
                f"gen={row['arc_gen_pass']}/{row['arc_gen_pass'] + row['arc_gen_fail']} "
                f"score={row['estimated_score']:.3f} size={row['filesize']} bytes time={row['seconds']:.1f}s",
                flush=True,
            )
        except Exception as exc:
            rows.append({
                "task": task_num,
                "onnx": str(path.relative_to(ROOT)),
                "ready": False,
                "error": repr(exc),
            })
            failures[task_num] = {"error": repr(exc), "traceback": traceback.format_exc()}
            print(f"  ERROR: {exc}", flush=True)
    df = pd.DataFrame(rows).sort_values("task").reset_index(drop=True)
    return df, failures

## 全 ONNX を検証

task8/task9 は重め。必要なら `evaluate_all([OUTPUT_DIR / "task001.onnx"])` のように個別実行する。

In [3]:
summary, failures = evaluate_all()
summary

task 001: validating task001.onnx ...
  ready=True agi=6/6 gen=262/262 score=15.272 size=3374 bytes time=0.8s
task 002: validating task002.onnx ...
  ready=True agi=6/6 gen=262/262 score=11.553 size=13751 bytes time=5.8s
task 003: validating task003.onnx ...
  ready=True agi=4/4 gen=261/261 score=13.771 size=10277 bytes time=3.2s
task 004: validating task004.onnx ...


2026-06-10 14:01:27.129 python[20774:10760777] 2026-06-10 14:01:27.129076 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_3'. It is not used by any node and should be removed from the model.


  ready=True agi=3/3 gen=262/262 score=12.113 size=31356 bytes time=6.8s
task 005: validating task005.onnx ...


2026-06-10 14:01:34.289 python[20774:10760777] 2026-06-10 14:01:34.289024 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_3'. It is not used by any node and should be removed from the model.


  ready=True agi=4/4 gen=262/262 score=10.245 size=105052 bytes time=17.5s
task 006: validating task006.onnx ...
  ready=True agi=4/4 gen=262/262 score=14.497 size=1540 bytes time=0.4s
task 007: validating task007.onnx ...
  ready=True agi=4/4 gen=262/262 score=12.356 size=21339 bytes time=3.2s
task 008: validating task008.onnx ...


2026-06-10 14:02:18.915 python[20774:10760777] 2026-06-10 14:02:18.914952 [E:onnxruntime:, profiler.cc:93 EndTimeAndRecordEvent] Maximum number of events reached, could not record profile event.


  ready=True agi=4/4 gen=262/262 score=9.600 size=1013473 bytes time=99.3s
task 009: validating task009.onnx ...


2026-06-10 14:03:41.070 python[20774:10760777] 2026-06-10 14:03:41.068702 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_0'. It is not used by any node and should be removed from the model.
2026-06-10 14:03:41.070 python[20774:10760777] 2026-06-10 14:03:41.069024 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_1'. It is not used by any node and should be removed from the model.
2026-06-10 14:04:05.500 python[20774:10760777] 2026-06-10 14:04:05.498349 [E:onnxruntime:, profiler.cc:93 EndTimeAndRecordEvent] Maximum number of events reached, could not record profile event.


  ready=True agi=4/4 gen=261/261 score=8.838 size=1405836 bytes time=208.7s
task 010: validating task010.onnx ...
  ready=True agi=3/3 gen=262/262 score=12.156 size=66146 bytes time=10.4s


,task,onnx,filesize,filesize_mb,size_ok,arc_agi_pass,arc_agi_fail,arc_agi_skip,arc_gen_pass,arc_gen_fail,arc_gen_skip,ready,memory_bytes,params,estimated_score,profile_path,seconds
0,1,outputs/task001.onnx,3374,0.003218,True,6,0,0,262,0,0,True,16620,156,15.272295,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,0.818525
1,2,outputs/task002.onnx,13751,0.013114,True,6,0,0,262,0,0,True,691404,46,11.553454,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,5.788446
2,3,outputs/task003.onnx,10277,0.009801,True,4,0,0,261,0,0,True,74308,985,13.770858,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,3.187565
3,4,outputs/task004.onnx,31356,0.029903,True,3,0,0,262,0,0,True,394040,1256,12.112610,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,6.849978
4,5,outputs/task005.onnx,105052,0.100185,True,4,0,0,262,0,0,True,2556860,1928,10.244956,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,17.458777
5,6,outputs/task006.onnx,1540,0.001469,True,4,0,0,262,0,0,True,36376,34,14.497401,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,0.370217
6,7,outputs/task007.onnx,21339,0.020350,True,4,0,0,262,0,0,True,307084,2809,12.356018,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,3.163912
7,8,outputs/task008.onnx,1013473,0.966523,True,4,0,0,262,0,0,True,4766016,111038,9.599948,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,99.252549
8,9,outputs/task009.onnx,1405836,1.340710,True,4,0,0,261,0,0,True,10358168,90122,8.838051,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,208.668405
9,10,outputs/task010.onnx,66146,0.063082,True,3,0,0,262,0,0,True,369356,9050,12.156277,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,10.360833


In [4]:
display_cols = [
    "task", "ready", "estimated_score", "memory_bytes", "params", "filesize", "size_ok",
    "arc_agi_pass", "arc_agi_fail", "arc_gen_pass", "arc_gen_fail", "seconds",
]
summary[display_cols].style.format({
    "estimated_score": "{:.3f}",
    "filesize": "{:,}",
    "memory_bytes": "{:,}",
    "params": "{:,}",
    "seconds": "{:.1f}",
})

,task,ready,estimated_score,memory_bytes,params,filesize,size_ok,arc_agi_pass,arc_agi_fail,arc_gen_pass,arc_gen_fail,seconds
0,1,True,15.272,"16,620",156,"3,374",True,6,0,262,0,0.8
1,2,True,11.553,"691,404",46,"13,751",True,6,0,262,0,5.8
2,3,True,13.771,"74,308",985,"10,277",True,4,0,261,0,3.2
3,4,True,12.113,"394,040","1,256","31,356",True,3,0,262,0,6.8
4,5,True,10.245,"2,556,860","1,928","105,052",True,4,0,262,0,17.5
5,6,True,14.497,"36,376",34,"1,540",True,4,0,262,0,0.4
6,7,True,12.356,"307,084","2,809","21,339",True,4,0,262,0,3.2
7,8,True,9.600,"4,766,016","111,038","1,013,473",True,4,0,262,0,99.3
8,9,True,8.838,"10,358,168","90,122","1,405,836",True,4,0,261,0,208.7
9,10,True,12.156,"369,356","9,050","66,146",True,3,0,262,0,10.4


In [5]:
total_estimated_score = summary["estimated_score"].dropna().sum()
ready_count = int(summary["ready"].fillna(False).sum())
print(f"ready tasks: {ready_count}/{len(summary)}")
print(f"total estimated score: {total_estimated_score:.3f}")
print(f"filesize limit: {FILESIZE_LIMIT:,} bytes")

ready tasks: 10/10
total estimated score: 120.402
filesize limit: 1,509,949 bytes


## 失敗例の確認

`failures` に最初の失敗例を保存している。空なら全 task pass。

In [6]:
failures

{}

In [7]:
# 個別 task の再検証例
# one, one_failures = evaluate_all([OUTPUT_DIR / "task009.onnx"])
# one